swin transformer

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import os
from collections import Counter
import numpy as np
import math

def main():
    # -----------------------------
    # 1. 設定パラメータ
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    BATCH_SIZE = 8
    NUM_EPOCHS = 10
    LEARNING_RATE = 0.001 # ViT系は学習率に敏感なので、収束しない場合は 0.0001 等に下げてください
    NUM_CLASSES = 3
    
    # 学習させたいデータ総数のリスト
    TRAIN_SIZES = [50]
    # TRAIN_SIZES = [50,60,70,80,90, 100,110,120,130,140, 150,160,170,180,190, 200]

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # -----------------------------
    # 2. データの前処理と読み込み
    # -----------------------------
    # Swin Transformerも基本は224x224でOKです
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
    class_names = full_dataset.classes
    all_targets = np.array(full_dataset.targets)
    
    print(f"検知されたクラス: {class_names}") 

    # 保存先ディレクトリの作成（念のため）
    os.makedirs('swintransformer', exist_ok=True)

    # =========================================================
    #  ここからサイズごとのループ実行
    # =========================================================
    for total_train_size in TRAIN_SIZES:
        print(f"\n##################################################")
        print(f"  開始(Swin-T Scratch): 学習データ数 {total_train_size} 枚")
        print(f"##################################################")

        # ---------------------------------------------------------
        # クラスごとの枚数を計算
        # ---------------------------------------------------------
        base_count = total_train_size // NUM_CLASSES
        remainder = total_train_size % NUM_CLASSES
        
        target_counts = {}
        for i in range(NUM_CLASSES):
            add = 1 if i < remainder else 0
            target_counts[i] = base_count + add

        print(f"目標内訳: {target_counts}")

        # ---------------------------------------------------------
        # データ分割処理
        # ---------------------------------------------------------
        train_indices = []
        val_indices = []
        
        # ★重要: シード固定
        np.random.seed(42)

        for class_idx in range(NUM_CLASSES):
            indices_of_class = np.where(all_targets == class_idx)[0]
            np.random.shuffle(indices_of_class)
            
            count = target_counts[class_idx]
            
            if len(indices_of_class) < count:
                 raise ValueError(f"クラス {class_names[class_idx]} データ不足")

            train_indices.extend(indices_of_class[:count])
            val_indices.extend(indices_of_class[count:])
        
        train_dataset = Subset(full_dataset, train_indices)
        val_dataset = Subset(full_dataset, val_indices)

        # ---------------------------------------------------------
        # 分割情報の保存
        # ---------------------------------------------------------
        split_filename = f'swintransformer/dataset_split_swin_{total_train_size}.pth'
        split_info = {
            'train_indices': train_indices,
            'val_indices': val_indices,
            'class_names': class_names,
            'total_size': total_train_size
        }
        torch.save(split_info, split_filename)

        train_size_real = len(train_dataset)
        val_size_real = len(val_dataset)
        
        # DataLoader作成
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # -----------------------------
        # 3. モデルの構築 (Swin Transformer)
        # -----------------------------
        # weights=None でスクラッチ学習
        # swin_t (Tiny) は ResNet50 とパラメータ数が近く比較に適しています
        model = models.swin_t(weights=None)
        
        # 【重要変更点】Swin Transformer (torchvision版) は fc ではなく head です
        num_ftrs = model.head.in_features
        model.head = nn.Linear(num_ftrs, NUM_CLASSES)
        
        model = model.to(device)

        # -----------------------------
        # 4. 損失関数と最適化手法
        # -----------------------------
        criterion = nn.CrossEntropyLoss()
        
        # Transformer系は AdamW が推奨されることが多いですが、
        # 比較のため Adam のままでも動きます。収束が悪ければ AdamW に変更してください。
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        # optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05) # 推奨設定例

        # -----------------------------
        # 5. 学習ループ
        # -----------------------------
        for epoch in range(NUM_EPOCHS):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            # Training
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

            epoch_loss = running_loss / train_size_real
            epoch_acc = correct / total

            # Validation
            model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    
                    val_loss += loss.item() * inputs.size(0)
                    _, predicted = torch.max(outputs, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()

            val_epoch_loss = val_loss / val_size_real
            val_epoch_acc = val_correct / val_total
            
            print(f'[{total_train_size}枚-Swin] Epoch {epoch+1}/{NUM_EPOCHS} '
                  f'Train Acc: {epoch_acc:.4f} | Val Acc: {val_epoch_acc:.4f}')

        # -----------------------------
        # 6. モデルの保存
        # -----------------------------
        model_filename = f'swintransformer/swin_{total_train_size}.pth'
        torch.save(model.state_dict(), model_filename)
        print(f"★ モデル保存完了: {model_filename}\n")

    print("全てのパターンの学習(Swin Scratch)が完了しました。")

if __name__ == '__main__':
    main()

使用デバイス: cuda:0
検知されたクラス: ['A', 'B', 'C']

##################################################
  開始(Swin-T Scratch): 学習データ数 50 枚
##################################################
目標内訳: {0: 17, 1: 17, 2: 16}
[50枚-Swin] Epoch 1/10 Train Acc: 0.4200 | Val Acc: 0.4848
[50枚-Swin] Epoch 2/10 Train Acc: 0.3800 | Val Acc: 0.2966
[50枚-Swin] Epoch 3/10 Train Acc: 0.3800 | Val Acc: 0.2449
[50枚-Swin] Epoch 4/10 Train Acc: 0.3600 | Val Acc: 0.3796
[50枚-Swin] Epoch 5/10 Train Acc: 0.3200 | Val Acc: 0.3289
[50枚-Swin] Epoch 6/10 Train Acc: 0.3800 | Val Acc: 0.4585
[50枚-Swin] Epoch 7/10 Train Acc: 0.2800 | Val Acc: 0.1984
[50枚-Swin] Epoch 8/10 Train Acc: 0.3400 | Val Acc: 0.2328
[50枚-Swin] Epoch 9/10 Train Acc: 0.3600 | Val Acc: 0.4747
[50枚-Swin] Epoch 10/10 Train Acc: 0.3000 | Val Acc: 0.4838
★ モデル保存完了: swintransformer/swin_50.pth

全てのパターンの学習(Swin Scratch)が完了しました。


In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

def main():
    # -----------------------------
    # 1. 設定
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    # 先ほどの学習コードで保存したディレクトリ
    SAVE_DIR = 'swintransformer' 
    TRAIN_SIZES = [50,60,70,80,90, 100,110,120,130,140, 150,160,170,180,190, 200]
    NUM_CLASSES = 3
    BATCH_SIZE = 8
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"評価デバイス: {device}")

    # -----------------------------
    # 2. データの準備 (共通)
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)

    # 結果を保存するリスト
    results = []

    # 保存先ディレクトリの確認（混同行列保存用）
    os.makedirs(SAVE_DIR, exist_ok=True)

    # -----------------------------
    # 3. ループで順にテスト
    # -----------------------------
    for size in TRAIN_SIZES:
        # パス設定 (Swin用に変更)
        split_file = os.path.join(SAVE_DIR, f'dataset_split_swin_{size}.pth')
        model_file = os.path.join(SAVE_DIR, f'swin_{size}.pth')

        if not os.path.exists(split_file) or not os.path.exists(model_file):
            print(f"スキップ: {size}枚のファイルが見つかりません。")
            continue

        print(f"\n##################################################")
        print(f"  評価開始(Swin): 学習データ {size} 枚モデル")
        print(f"##################################################")

        # (1) 分割情報のロード
        # weights_only=False はPyTorchのバージョンによっては不要ですが、念のため入れています
        try:
            split_info = torch.load(split_file, weights_only=False)
        except TypeError:
             # 古いPyTorch対応
            split_info = torch.load(split_file)
            
        val_indices = split_info['val_indices']
        class_names = split_info['class_names'] # クラス名を取得
        
        # 検証セットを復元
        val_dataset = Subset(full_dataset, val_indices)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # (2) モデルのロード (Swin用に変更)
        model = models.swin_t(weights=None)
        
        # Swinの出力層は 'head' です
        num_ftrs = model.head.in_features
        model.head = nn.Linear(num_ftrs, NUM_CLASSES)
        
        model.load_state_dict(torch.load(model_file, map_location=device))
        model = model.to(device)
        model.eval()

        # (3) 推論
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # (4) 詳細レポート出力
        print(f"\n--- Classification Report (Size: {size}) ---")
        # zero_division=0 を追加して、予測されなかったクラスの警告を抑制
        report = classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)
        print(report)

        # (5) 混同行列の作成と保存
        cm = confusion_matrix(all_labels, all_preds)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix (Swin Train Size: {size})')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        
        # 画像として保存 (ファイル名重複防止のため swin_ を付与)
        save_img_name = os.path.join(SAVE_DIR, f'swin_confusion_matrix_{size}.png')
        plt.savefig(save_img_name)
        plt.close() # メモリ解放
        print(f" >> 混同行列を保存しました: {save_img_name}")

        # (6) 結果リストに追加
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')
        
        results.append({
            "Train Size": size,
            "Accuracy": acc,
            "Macro F1": f1,
            "Val Data Count": len(val_indices)
        })

    # -----------------------------
    # 4. 結果まとめ表示
    # -----------------------------
    print("\n\n=== [Swin] データ数ごとの精度比較まとめ ===")
    df = pd.DataFrame(results)
    # 見やすいように丸める
    pd.options.display.float_format = '{:.4f}'.format
    print(df.to_string(index=False))

    # CSVにも保存
    csv_save_path = os.path.join(SAVE_DIR, "swin_accuracy_comparison.csv")
    df.to_csv(csv_save_path, index=False)
    print(f"\nまとめ結果を '{csv_save_path}' に保存しました。")

if __name__ == '__main__':
    main()

評価デバイス: cuda:0

##################################################
  評価開始(Swin): 学習データ 50 枚モデル
##################################################

--- Classification Report (Size: 50) ---
              precision    recall  f1-score   support

           A       0.48      1.00      0.65       479
           B       0.00      0.00      0.00       285
           C       0.00      0.00      0.00       224

    accuracy                           0.48       988
   macro avg       0.16      0.33      0.22       988
weighted avg       0.23      0.48      0.32       988

 >> 混同行列を保存しました: swintransformer/swin_confusion_matrix_50.png
スキップ: 60枚のファイルが見つかりません。
スキップ: 70枚のファイルが見つかりません。
スキップ: 80枚のファイルが見つかりません。
スキップ: 90枚のファイルが見つかりません。
スキップ: 100枚のファイルが見つかりません。
スキップ: 110枚のファイルが見つかりません。
スキップ: 120枚のファイルが見つかりません。
スキップ: 130枚のファイルが見つかりません。
スキップ: 140枚のファイルが見つかりません。
スキップ: 150枚のファイルが見つかりません。
スキップ: 160枚のファイルが見つかりません。
スキップ: 170枚のファイルが見つかりません。
スキップ: 180枚のファイルが見つかりません。
スキップ: 190枚のファイルが見つかりません。
スキップ: 200枚のファイルが見つかりません